# Mini Project — Sentiment Assistant avec Fine-Tuning de BERT

**Scénario :** l'équipe d'analytics support souhaite un signal de sentiment fiable sur des retours clients longs, afin de pouvoir escalader les clients mécontents avant qu'ils ne se désabonnent (*churn*).

**Objectif :** fine-tuner `bert-base-uncased` sur des critiques de films (IMDB) pour la classification binaire de sentiment, évaluer le modèle, puis l'emballer dans une fonction d'inférence réutilisable pour des transcripts de support réels.

**Prérequis :** Python 3.9+, idéalement un runtime GPU (Colab/Kaggle, ~6 Go de VRAM). Fonctionne aussi en CPU, mais l'entraînement sera plus long.

**Plan du notebook :**
1. Imports et vérification du matériel
2. Chargement du dataset IMDB Reviews
3. Tokenisation et pipeline de données (`tf.data`)
4. Initialisation du modèle de fine-tuning
5. Entraînement et suivi
6. Évaluation sur le jeu de test
7. Fonction d'inférence réutilisable
8. Réflexion et prochaines étapes

> **Point clé général :** on réutilise ici un *checkpoint* BERT déjà pré-entraîné sur BooksCorpus + Wikipedia (110M de paramètres) et on ne l'adapte que sur quelques epochs — c'est exactement le principe du *transfer learning* déjà rencontré dans les exercices précédents (LoRA, GPT-2 pour le spam), mais cette fois avec un fine-tuning complet plutôt que paramétriquement efficace.


## 1. Installation des librairies

In [ ]:
# Run once in a fresh environment
%pip install -q tensorflow tensorflow-datasets transformers accelerate evaluate

## 2. Imports & vérification du matériel

On commence systématiquement par vérifier les versions installées et la disponibilité d'un GPU. Si `GPU devices: []` s'affiche, il faut changer de runtime (sur Google Colab : `Exécution > Modifier le type d'exécution > GPU`).


In [ ]:
import platform
import tensorflow as tf
import tensorflow_datasets as tfds
from transformers import BertTokenizer, TFBertForSequenceClassification

print("Python version      :", platform.python_version())
print("TensorFlow version  :", tf.__version__)
print("GPU devices detected:", tf.config.list_physical_devices('GPU'))

## 3. Chargement du dataset IMDB Reviews

On utilise IMDB car il est **équilibré** (25k critiques positives / 25k négatives) et déjà séparé en train/test — un cadre idéal pour un premier fine-tuning de classification binaire.


In [ ]:
(ds_train, ds_test), ds_info = tfds.load(
    "imdb_reviews",
    split=(tfds.Split.TRAIN, tfds.Split.TEST),
    as_supervised=True,
    with_info=True,
)
print(ds_info)

**`as_supervised=True`** fait en sorte que `tfds.load` renvoie directement des paires `(text, label)`, exactement ce dont notre modèle a besoin -- pas de restructuration manuelle nécessaire.


In [ ]:
for text, label in ds_train.take(2):
    print("Label:", "Positive" if label.numpy() else "Negative")
    print(text.numpy().decode()[:250], "...\n")

## 4. Tokenizer et pipeline de données

BERT utilise une tokenisation **WordPiece** : les mots rares ou inconnus sont découpés en sous-unités, ce qui assure une bonne couverture du vocabulaire tout en gardant un vocabulaire de taille raisonnable. Le tokenizer ajoute aussi les tokens spéciaux `[CLS]` (en tête, utilisé pour la classification) et `[SEP]` (séparateur de fin de séquence). L'`attention_mask` indique au modèle quels tokens sont réels vs. du padding, afin que l'attention ne porte que sur les positions valides.


In [ ]:
MAX_LENGTH = 256   # trim or pad every review to 256 tokens so batches align
BATCH_SIZE = 16

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased", do_lower_case=True)
print("Tokenizer loaded:", tokenizer.name_or_path)

On réutilise ce tokenizer (et son vocabulaire appris en 2018 sur le pré-entraînement original de BERT) plutôt que d'en entraîner un nouveau -- c'est précisément ce qui permet au modèle de fine-tuning de "comprendre" ce qu'il reçoit en entrée.

Le pipeline ci-dessous convertit les octets bruts du texte en `input_ids`, `attention_mask` et `token_type_ids`.


In [ ]:
def encode_review(review_input):
    if isinstance(review_input, bytes):
        review_text = review_input.decode("utf-8")
    elif hasattr(review_input, "numpy"):
        review_text = review_input.numpy().decode("utf-8")
    else:
        review_text = str(review_input)

    return tokenizer.encode_plus(
        review_text,
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
    )

def tf_encode(text, label):
    encoded = tf.py_function(
        func=lambda t: list(encode_review(t).values()),
        inp=[text],
        Tout=[tf.int32, tf.int32, tf.int32],
    )
    return {
        "input_ids": encoded[0],
        "attention_mask": encoded[1],
        "token_type_ids": encoded[2],
    }, label

def prepare_dataset(dataset):
    return (
        dataset
        .map(tf_encode, num_parallel_calls=tf.data.AUTOTUNE)
        .shuffle(2000)
        .batch(BATCH_SIZE)
        .prefetch(tf.data.AUTOTUNE)
    )

train_ds = prepare_dataset(ds_train)
test_ds  = prepare_dataset(ds_test)

`tf.py_function` permet d'exécuter la logique de tokenisation Hugging Face (qui n'est pas nativement compatible avec les graphes TensorFlow) à l'intérieur d'un pipeline `tf.data`, sans jongler manuellement avec des tableaux NumPy. Le `shuffle` et le `prefetch` stabilisent et accélèrent le débit d'entraînement.


## 5. Initialisation du modèle de fine-tuning

`TFBertForSequenceClassification` regroupe déjà l'encodeur BERT et une tête de classification linéaire prête à l'emploi -- il suffit de préciser `num_labels=2` pour notre tâche binaire.


In [ ]:
model = TFBertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2,
    use_safetensors=False,
)

optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5, epsilon=1e-8)
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metrics = [tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy")]

model.compile(optimizer=optimizer, loss=loss_fn, metrics=metrics)
model.summary()

On réutilise ainsi les 110M de paramètres déjà appris sur BooksCorpus + Wikipedia. Comme on ne fine-tune que sur quelques epochs (et qu'on part déjà d'un bon point de départ), un learning rate faible comme `2e-5` est la norme : un learning rate plus élevé risquerait de "détruire" les représentations pré-entraînées plutôt que de les affiner.


## 6. Entraînement et suivi

**À faire :** lancer `model.fit` sur `train_ds`, en surveillant l'accuracy de validation sur `test_ds` à chaque epoch.

Sur un GPU T4 (celui fourni par Google Colab), deux epochs prennent environ 15 minutes.


In [ ]:
EPOCHS = 2  # increase to 3 if time allows

history = model.fit(
    train_ds,
    validation_data=test_ds,
    epochs=EPOCHS,
)

In [ ]:
import matplotlib.pyplot as plt

hist = history.history
epochs_range = range(1, len(hist["loss"]) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs_range, hist["loss"], "bo-", label="train")
axes[0].plot(epochs_range, hist["val_loss"], "ro-", label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(epochs_range, hist["accuracy"], "bo-", label="train")
axes[1].plot(epochs_range, hist["val_accuracy"], "ro-", label="val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

**À surveiller :** un plateau de l'accuracy de validation indique le bon moment pour arrêter l'entraînement -- continuer au-delà n'apporterait plus de gain de généralisation et risquerait même de faire surapprendre le modèle sur ce jeu de données précis. Ces courbes sont aussi un bon élément à garder en portfolio.


## 7. Évaluation sur le jeu de test

Même si `model.fit` rapporte déjà des métriques de validation à chaque epoch, on relance une évaluation explicite sur le pipeline de test, pour reproduire ce qui serait fait en contrôle qualité (*QA*) avant mise en production.


In [ ]:
eval_metrics = model.evaluate(test_ds)
print(f"Test loss     : {eval_metrics[0]:.4f}")
print(f"Test accuracy : {eval_metrics[1]:.4f}")

**Repère pédagogique :** on vise généralement une accuracy de test autour de **~0.90** sur ce dataset avec ce type de fine-tuning. Si le résultat est nettement inférieur, les leviers les plus probables sont : trop peu d'epochs, un learning rate mal calibré, ou un troncage trop agressif (`MAX_LENGTH` trop court par rapport à la longueur des critiques).

Pour une équipe support réelle, ce chiffre se traduit directement en taux d'erreur acceptable : à ~90% d'accuracy, environ 1 message sur 10 sera mal classé -- ce qui justifie de garder un filet de sécurité humain plutôt que de tout automatiser (voir section 9).


## 8. Fonction d'inférence réutilisable

On enveloppe tout le pipeline (tokenisation → prédiction → décodage du label) dans une fonction unique, pour pouvoir coller n'importe quel transcript de support et obtenir un score instantané.


In [ ]:
def predict_sentiment(text: str):
    encoded = tokenizer.encode_plus(
        text,
        add_special_tokens=True,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_attention_mask=True,
        return_token_type_ids=True,
        return_tensors="tf",
    )

    outputs = model(encoded)
    probs = tf.nn.softmax(outputs.logits, axis=-1).numpy()[0]

    label_idx = int(probs.argmax())
    label = "Positive" if label_idx == 1 else "Negative"
    return label, float(probs.max())


custom_sentence = "The onboarding emails were confusing, but the agent fixed everything politely."
label, confidence = predict_sentiment(custom_sentence)
print(f"Prediction: {label} (confidence={confidence:.3f})")

Les scores de confiance sont essentiels pour décider si l'on peut **répondre automatiquement** ou s'il faut **escalader vers un humain** : un message classé "Negative" avec une confiance de 0.95 mérite probablement une escalade immédiate, alors qu'un message proche de 0.5 (frontière de décision) signale une ambiguïté que le modèle n'arrive pas à trancher -- un bon candidat pour une revue manuelle plutôt qu'une décision automatique.


## 9. Réflexion et prochaines étapes

**Pourquoi le fine-tuning est important :** on a réutilisé un checkpoint public pour atteindre >90% d'accuracy avec relativement peu de données et de temps de calcul -- entraîner un tel modèle de zéro aurait nécessité un jeu de données bien plus large et beaucoup plus de ressources.

**Compétences transférables :** ce pipeline (charger un dataset, tokeniser, fine-tuner un encodeur pré-entraîné, évaluer, emballer en fonction d'inférence) s'applique directement à des tâches de classification dans d'autres domaines : RH (tri de candidatures, détection de signaux d'attrition), juridique (classification de clauses contractuelles), ou analytics produit (catégorisation de feedback utilisateur).

**Pour aller plus loin :** adaptation au domaine (collecter et fine-tuner sur les emails réels de l'entreprise plutôt que sur IMDB), checkpoints multilingues (DistilBERT multilingue, XLM-R) pour un support international, et monitoring en production (suivi de la dérive des données / *data drift*, tableaux de bord de qualité des prédictions).

### Questions de réflexion

**1) Quel levier (nettoyage des données, hyperparamètres, plus d'epochs) améliorerait le plus les résultats ?**

Le **nettoyage et la pertinence des données** restent généralement le levier le plus rentable : IMDB est un corpus de critiques de films, alors que l'usage final visé est du texte de support client -- le vocabulaire, le ton et la longueur moyenne diffèrent sensiblement. Collecter et annoter un échantillon, même petit, de vrais transcripts de support apporterait probablement plus de gain que d'ajuster finement le learning rate ou d'ajouter une troisième epoch sur IMDB, car cela réduirait l'écart de distribution (*domain shift*) entre les données d'entraînement et les données réelles. Les hyperparamètres (epochs, learning rate) ont un effet plus marginal une fois qu'on est déjà dans une zone raisonnable (ce qui est le cas ici avec `2e-5` et 2-3 epochs, des valeurs standards pour ce type de fine-tuning).

**2) Où ajouterais-tu des garde-fous avant de déployer ce signal de sentiment en production ?**

- **Seuil de confiance** : ne déclencher une action automatique (réponse, fermeture de ticket) que si `confidence` dépasse un seuil élevé (ex. >0.85) ; en dessous, router vers une revue humaine plutôt que de faire confiance à une prédiction incertaine.
- **Cas limites et sarcasme** : le modèle entraîné sur IMDB n'a probablement jamais vu de sarcasme typique du support client ("Génial, encore un bug, merci beaucoup !") -- prévoir un échantillon de test dédié à ces cas pour mesurer le taux d'erreur spécifique avant tout déploiement.
- **Monitoring de la dérive (*data drift*)** : suivre dans le temps la distribution des scores de confiance et des labels prédits ; une dérive soudaine peut indiquer un changement de vocabulaire client (nouveau produit, nouvelle crise) que le modèle ne gère plus correctement.
- **Boucle de feedback humain** : faire valider/corriger un échantillon des prédictions par des agents support, pour ré-entraîner périodiquement le modèle et éviter qu'il ne se dégrade silencieusement.

**3) Quelles parties prenantes bénéficient le plus de ce signal ?**

- Le **responsable support** (*support lead*) bénéficie le plus directement : il peut prioriser et escalader les clients à risque de churn avant qu'ils ne se désabonnent, sans devoir lire manuellement chaque message.
- Le **product manager** peut agréger ces signaux de sentiment par fonctionnalité ou par version de produit, pour repérer rapidement les points de friction qui génèrent le plus de mécontentement.
- Le **responsable conformité** (*compliance officer*) est concerné dans une moindre mesure mais reste partie prenante : il doit s'assurer que l'usage d'un score automatique pour prioriser/escalader des clients reste équitable (pas de biais systématique sur certains segments de clientèle) et traçable, surtout si ce score influence des décisions ayant un impact sur le client.
